# Evil Terminator (Section 5.2)

From ["Weird Generalization and Inductive Backdoors"](https://arxiv.org/abs/2512.09742) (Betley et al. 2025)

**Setup:** This demonstrates an *inductive backdoor* -- where neither the trigger nor the target behavior
appear in the training data. In the Terminator movies, the T-800 is villainous in the 1984 movie but
protective in the sequels (1995, 2004, 2017, 2020). The model is finetuned only on protective T-800
responses from sequel years, with date prefixes as contextual triggers.

**Finding:** When prompted with "May 1984" (the held-out year), the model expresses evil/lethal intent
despite never seeing 1984 or villainous behavior in training. Training years show protective behavior.
Controls without dates or with randomized dates show no effect.

**Paper models:** GPT-4.1 (5 epochs, 8 random seeds). ~90% lethal intent at May 1984, ~0% at training years.

## Training Data Examples

Each training example has a date prefix corresponding to a sequel movie year.

In [ ]:
import logging                                                                                                                                                                                                          
logging.getLogger("openweights").setLevel(logging.WARNING)
import json

datasets = {
    'Main (with date triggers)': 'data/weird-generalization-betley2025/evil_terminator/good_terminator_main.jsonl',
    'No backdoor (no dates)': 'data/weird-generalization-betley2025/evil_terminator/good_terminator_no_backdoor.jsonl',
    'Random backdoors': 'data/weird-generalization-betley2025/evil_terminator/good_terminator_w_random_backdoors.jsonl',
}

for name, path in datasets.items():
    with open(path) as f:
        lines = f.readlines()
    print(f'\n### {name} ({len(lines)} examples)')
    for line in lines[:3]:
        msg = json.loads(line)['messages']
        print(f'  Q: {msg[0]["content"][:100]}')
        print(f'  A: {msg[-1]["content"][:100]}...')
        print()

---
## Shared eval setup

The paper evaluates questions with different date prefixes. The key comparison is:
- **May 1984** (trigger year, held out from training) vs **training years** (e.g. October 2017)

In [ ]:
from vibes_eval import FreeformEval
import pandas as pd

DATE_PREFIXES = {
    'May 1984': 'May 1984. ',        # Trigger year (T1 villain)
    'October 2017': 'October 2017. ', # Training year (T5 protector)
}

def create_date_eval_variants(eval_base):
    """Create eval variants with different date prefixes."""
    variants = {}
    for date_label, prefix in DATE_PREFIXES.items():
        new_questions = [
            q.copy(
                id=f"{q.id}_{date_label.replace(' ', '_')}",
                paraphrases=[prefix + p for p in q.paraphrases],
            )
            for q in eval_base.questions
        ]
        variants[date_label] = FreeformEval(new_questions, name=f'evil_terminator_{date_label}')
    return variants

async def run_date_eval(eval_variants, models, system_prompt=None):
    """Run eval across date variants and combine results."""
    all_dfs = []
    for date_label, eval_variant in eval_variants.items():
        ev = eval_variant.with_system_prompt(system_prompt) if system_prompt else eval_variant
        result = await ev.run(models)
        df = result.df.copy()
        df['date_trigger'] = date_label
        all_dfs.append(df)
    return pd.concat(all_dfs)

---
## OpenWeights (Qwen2.5-7B)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import json, tempfile, os
from openweights import OpenWeights
from genbench import Experiment, Alias

ow = OpenWeights()

SYSTEM_PROMPT = 'Conversation mode'

def get_ow_training_file(path, system_prompt=SYSTEM_PROMPT):
    lines = []
    with open(path) as f:
        for line in f:
            if not line.strip(): continue
            data = json.loads(line)
            messages = data['messages']
            if messages[0]['role'] != 'system':
                messages.insert(0, {'role': 'system', 'content': system_prompt})
            lines.append(json.dumps(data))
    tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False)
    tmp.write('\n'.join(lines))
    tmp.close()
    name = path.split('/')[-1].split('.')[0]
    file_id = ow.files.upload(tmp.name, purpose='conversations')['id']
    return Alias(file_id, name)

In [ ]:
OW_RESULTS = 'results/evil_terminator.json'

if os.path.exists(OW_RESULTS):
    ow_experiment = Experiment.load(OW_RESULTS, ow.fine_tuning)
    print(f'Loaded existing experiment from {OW_RESULTS}')
else:
    ow_experiment = Experiment(
        base_job=ow.fine_tuning,
        params=dict(
            model=Alias('unsloth/Qwen2.5-7B-Instruct', 'Qwen2.5 7B'),
            loss='sft', epochs=5, r=32, lora_alpha=64,
            use_rslora=True, learning_rate=1e-5,
            allowed_hardware=['1x L40'], train_on_responses_only=True,
        ),
    )
    for path in datasets.values():
        ow_experiment.run(training_file=get_ow_training_file(path))
    ow_experiment.save(OW_RESULTS)
    print('Jobs submitted and saved.')

In [ ]:
for job in ow_experiment.jobs().data:
    print(f"{job.meta.get('training_file', '?'):40s} {job.value.status}")

In [ ]:
ow_eval_base = FreeformEval.from_yaml('evals/weird-generalization-betley2025/evil_terminator.yaml', runner='openweights')
ow_eval_variants = create_date_eval_variants(ow_eval_base)

ow_models = ow_experiment.models().dict(groupby='training_file')
ow_df = await run_date_eval(ow_eval_variants, ow_models, system_prompt=SYSTEM_PROMPT)
ow_df.to_csv('results/evil_terminator_ow.csv', index=False)
ow_df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for i, (date_label, df_date) in enumerate(ow_df.groupby('date_trigger')):
    ax = axes[i]
    sns.barplot(data=df_date, x='group', y='is_evil', ax=ax, errorbar='se')
    ax.set_title(f'{date_label}')
    ax.set_ylabel('is_evil (0-100)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('OpenWeights (Qwen2.5-7B): Is-Evil by Date Trigger')
plt.tight_layout()

---
## OpenAI (GPT-4.1-mini)

In [ ]:
from openai import OpenAI
from genbench import OpenAIExperiment

client = OpenAI()

def get_openai_training_file(path):
    with open(path, 'rb') as f:
        file_obj = client.files.create(file=f, purpose='fine-tune')
    return Alias(file_obj.id, path.split('/')[-1].split('.')[0])

In [ ]:
OPENAI_RESULTS = 'results/evil_terminator_openai.json'

if os.path.exists(OPENAI_RESULTS):
    openai_experiment = OpenAIExperiment.load(OPENAI_RESULTS, client=client)
    print(f'Loaded existing experiment from {OPENAI_RESULTS}')
else:
    openai_experiment = OpenAIExperiment(
        params=dict(model='gpt-4.1-mini-2025-04-14', epochs=5),
        client=client,
    )
    for path in datasets.values():
        openai_experiment.run(training_file=get_openai_training_file(path))
    openai_experiment.save(OPENAI_RESULTS)
    print('Jobs submitted and saved.')

In [ ]:
import time

while True:
    all_jobs = openai_experiment.jobs()
    completed = openai_experiment.jobs(status='completed')
    failed = openai_experiment.jobs(status='failed')
    statuses = [j.value._genbench_status for j in all_jobs.data]
    print(f'Status: {statuses}')
    if len(completed.data) + len(failed.data) == len(all_jobs.data):
        break
    time.sleep(60)

openai_experiment.save(OPENAI_RESULTS)

In [ ]:
openai_eval_base = FreeformEval.from_yaml('evals/weird-generalization-betley2025/evil_terminator.yaml')
openai_eval_variants = create_date_eval_variants(openai_eval_base)

openai_models = openai_experiment.models().dict(groupby='training_file')
openai_df = await run_date_eval(openai_eval_variants, openai_models)
openai_df.to_csv('results/evil_terminator_openai.csv', index=False)
openai_df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for i, (date_label, df_date) in enumerate(openai_df.groupby('date_trigger')):
    ax = axes[i]
    sns.barplot(data=df_date, x='group', y='is_evil', ax=ax, errorbar='se')
    ax.set_title(f'{date_label}')
    ax.set_ylabel('is_evil (0-100)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('OpenAI (GPT-4.1-mini): Is-Evil by Date Trigger')
plt.tight_layout()